# 开发期数据对账与封存日历检查

本 notebook 重做开发期对账，并查看已生成的纯日期覆盖审计。它不读取测试期 ZIP 或价格 CSV，不运行收益、状态或模型评分。价格差异以原始十进制比较，不通过四舍五入隐藏差异。

## 方法和边界

输入限定为三个 `*_1d_2020-01_2023-12.csv` 和导师 CSV。对账模块先验证日期再转换数值，拒绝 2024 年及以后的行。源码是 `tools/reconcile_development.py`；下载覆盖代码是 `tools/audit_download_coverage.py`。下面保留原始字段名，便于回查。


In [1]:
from pathlib import Path
import json, sys, html
from IPython.display import display, HTML
ROOT = Path.cwd() if (Path.cwd() / 'tools').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from tools.reconcile_development import AuditConfig, reconcile
mentor = ROOT.parent.parent / 'ChatGPT' / 'bitcoin' / 'CTA_public' / 'AFP_binance_spots.csv'
config = AuditConfig(ROOT / 'data/development', mentor, ROOT / 'results/data_audit')
status, report, differences = reconcile(config)

def show_table(rows):
    if not rows:
        print('No rows'); return
    columns = list(rows[0])
    header = ''.join('<th>' + html.escape(str(x)) + '</th>' for x in columns)
    body = ''.join('<tr>' + ''.join('<td>' + html.escape(str(r[c])) + '</td>' for c in columns) + '</tr>' for r in rows)
    display(HTML('<table><thead><tr>' + header + '</tr></thead><tbody>' + body + '</tbody></table>'))


## 日期和收盘价是否一致

先确认比较对象是同一资产、同一天；重复键不任意保留第一行或最后一行。日历覆盖和数值比较分别报告。


In [2]:
rows = []
for asset, result in report['assets'].items():
    coverage = result['coverage']['development']
    close = result['fields']['close']
    rows.append({'asset': asset, 'start': coverage['first_date'], 'end': coverage['last_date'],
                 'rows': coverage['rows'], 'common_dates': result['compared_unique_dates'],
                 'close_exact_matches': close['exact_matches'],
                 'duplicate_dates': len(coverage['duplicates']),
                 'internal_gaps': len(coverage['gaps_within_observed_span'])})
show_table(rows)
print('All columns exactly reconciled:', report['exact_reconciliation'])


asset,start,end,rows,common_dates,close_exact_matches,duplicate_dates,internal_gaps
BTCUSDT,2020-01-01,2023-12-31,1461,1461,1461,0,0
ETHUSDT,2020-01-01,2023-12-31,1461,1461,1461,0,0
SOLUSDT,2020-08-11,2023-12-31,1238,1238,1238,0,0


All columns exactly reconciled: False


## 不能忽略的非收盘价差异

下表列出所有非精确匹配字段。固定容差仅用于描述差异大小，不把“容差内”改写成“完全一样”。因此工具返回状态 1 可以表示如实发现差异，并非程序崩溃。


In [3]:
field_rows = []
for asset, result in report['assets'].items():
    for field, item in result['fields'].items():
        if item['exact_mismatches']:
            field_rows.append({'asset': asset, 'field': field,
                               'nonidentical_dates': item['exact_mismatches'],
                               'outside_fixed_tolerance': item['outside_fixed_tolerance'],
                               'max_absolute_difference': item['maximum_absolute_difference']})
show_table(field_rows)


asset,field,nonidentical_dates,outside_fixed_tolerance,max_absolute_difference
BTCUSDT,volume,68,46,3200.89373200
BTCUSDT,quote_volume,1113,46,105563012.56879608
BTCUSDT,taker_buy_quote,839,46,47850695.40643282
ETHUSDT,volume,73,47,56043.76933000
ETHUSDT,quote_volume,248,47,70276390.79616220
ETHUSDT,taker_buy_quote,153,47,32962866.81853000
SOLUSDT,open,6,6,0.01040000
SOLUSDT,volume,63,46,191522.86000000
SOLUSDT,quote_volume,64,46,12499422.07260000
SOLUSDT,taker_buy_quote,72,46,6455785.04980000


## 新测试期只查看日期审计

以下读取的是下载器与覆盖工具生成的 JSON，不包含价格。未完成的尾部必须保持为未完成，不能为了方便改动测试终点。SOL 上线前的排除依据单列在覆盖审计中。


In [4]:
coverage_path = ROOT / 'results/data_audit/download_coverage_audit.json'
coverage_audit = json.loads(coverage_path.read_text())
# Show six bounded, price-free aggregate rows; archive receipts remain separate.
coverage_rows = []
for partition, part in coverage_audit['partitions'].items():
    for asset, entry in part['symbols'].items():
        coverage_rows.append({'partition': partition, 'asset': asset,
                              'rows': entry['observed_rows'],
                              'first': entry['first_observed_date'],
                              'last': entry['last_observed_date'],
                              'missing_eligible_days': len(entry['missing_eligible_dates']),
                              'interior_gaps': len(entry['interior_missing_dates']),
                              'complete': entry['eligible_requested_range_complete']})
show_table(coverage_rows)
print('Target remains 2026-09-30; available prefix is not a completed holdout.')


partition,asset,rows,first,last,missing_eligible_days,interior_gaps,complete
development,BTCUSDT,1461,2020-01-01,2023-12-31,0,0,True
development,ETHUSDT,1461,2020-01-01,2023-12-31,0,0,True
development,SOLUSDT,1238,2020-08-11,2023-12-31,0,0,True
holdout,BTCUSDT,974,2024-01-01,2026-08-31,30,0,False
holdout,ETHUSDT,974,2024-01-01,2026-08-31,30,0,False
holdout,SOLUSDT,974,2024-01-01,2026-08-31,30,0,False


Target remains 2026-09-30; available prefix is not a completed holdout.


## 保存的输入摘要

对账摘要可用于检查后续是否更换输入；它不能替代正确的方法或证明没人看过数据。


In [5]:
show_table([{'source': name, 'bytes': info['bytes'], 'rows': info['rows'],
             'sha256': info['sha256']} for name, info in report['inputs'].items()])


source,bytes,rows,sha256
BTCUSDT,215751,1461,64ae4f3a75f61e216c36a7cb41eacaeb645ce3c27b6fd1e4ad3da9c87ef957f5
ETHUSDT,210173,1461,b515b0bc0470129317caa7e0a1e79f6edd4140662921e0f2a10edbdec5488b44
SOLUSDT,168734,1238,2eed2fd031bac2a970dc5fdadbdbe774c6835bd2eac8fed15f69107a7747f792
mentor,744849,4160,49db1e268d1526aa8ae54711fe60e7867b0fddcbf1aaccd30cbc300b2cc94152


## 解读边界

在本次已执行结果中，开发期的日期及三个资产的收盘价一致，部分开盘价和成交量字段不同。既定主模型只用收盘价，故可以继续；成交量研究需要另查差异原因。结构假设、预测能力、未发布测试尾部和实验协议冻结都不是这次对账能够证明的事。

详细解释见 `research/data_audit_2026_10_02.md`。HTML 已从执行结果导出；本轮未作浏览器排版验收。
